# Day 4 — Unity Catalog: Fundamentals & Governed Access

Allow about 2.5–3.5 hours for this notebook (plus any lecture time). Work through the sections in order, or split blocks across the class if needed.

You will use the three-level name (`catalog.schema.object`), create a personal schema and several views over existing Day 3 Delta paths on ABFS, run metadata and analytics queries, and finish with challenges and SQL drills.

Prerequisites: Unity Catalog enabled, Day 3 Silver/Gold paths available, UC-compatible compute, and `%run ./02-Mount-Azure-Data-Lake`.

Related reference material (external courses, adapted here to ABFS and views): Databricks RBAC / lineage labs (M-7), workspace permissions (M-1), and security/governance examples (e.g. masked views).

## 1 — Connect ADLS (OAuth + `base_path`)

Same pattern as Days 1–3. Run the mount helper, then set paths.

In [ ]:
%run ./02-Mount-Azure-Data-Lake

In [ ]:
# Paths from %run
BASE_PATH = base_path
DAY03_SILVER = BASE_PATH + "/day03/silver/flights_clean"
DAY03_GOLD   = BASE_PATH + "/day03/gold/flights_by_destination"

# Unity Catalog naming — CHANGE STUDENT_ID to your assigned u01–u16
CATALOG    = "main"
STUDENT_ID = "u01"
SCHEMA_NAME = f"day04_{STUDENT_ID}_lab"
FULL_SCHEMA = f"{CATALOG}.{SCHEMA_NAME}"

print("Delta Silver :", DAY03_SILVER)
print("Delta Gold   :", DAY03_GOLD)
print("UC Schema    :", FULL_SCHEMA)

## 2 — Three-level namespace deep dive

Unity Catalog organizes every securable object in a **catalog → schema → object** hierarchy.

```
┌──────────────────────────────────────────────────────────┐
│  METASTORE  (regional — shared by workspaces)            │
│  ├── catalog: main                                       │
│  │   ├── schema: day04_u01_lab                           │
│  │   │   ├── VIEW  flights_silver_v                      │
│  │   │   ├── VIEW  flights_gold_v                        │
│  │   │   └── VIEW  flights_silver_dest_only_v            │
│  │   └── schema: default                                 │
│  ├── catalog: hive_metastore  (legacy)                   │
│  └── catalog: system          (metadata & audit)         │
└──────────────────────────────────────────────────────────┘
```

### Naming rules

| Level | Maps to | Naming in this lab |
|-------|---------|--------------------|
| **Catalog** | Environment / org boundary | `main` (or the catalog name your organization uses) |
| **Schema** | Database / project area | `day04_uXX_lab` (personal) |
| **Object** | Table, view, volume, function, model | `flights_silver_v`, etc. |

### Fully qualified name

```sql
SELECT * FROM main.day04_u01_lab.flights_silver_v
--           ^^^^  ^^^^^^^^^^^^^^  ^^^^^^^^^^^^^^^^
--           cat   schema          object
```

### 2b — `main` vs `hive_metastore` vs `system`

| Catalog | Purpose | You use it for |
|---------|---------|----------------|
| **`main`** | Default UC catalog; new objects go here | Creating schemas, views, tables |
| **`hive_metastore`** | Legacy; pre-UC tables live here | Backward compat; avoid for new work |
| **`system`** | Databricks system tables (audit, lineage, billing) | Querying `information_schema`, audit logs |

**Rule:** Unless told otherwise, always create objects in **`main`** (or your org's named catalog).  
**Warning:** Mixing UC + legacy `hive_metastore` objects in the same query may produce unexpected behavior.

In [ ]:
# Create your personal schema (idempotent)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FULL_SCHEMA} COMMENT 'Day 4 hands-on lab — UC fundamentals'")
print("✓ Schema ready:", FULL_SCHEMA)

In [ ]:
# Explore available catalogs
try:
    print("=== CATALOGS ===")
    spark.sql("SHOW CATALOGS").show(truncate=False)
except Exception as e:
    print("SHOW CATALOGS:", type(e).__name__, e)

In [ ]:
# Schemas in your catalog — spot your new schema
try:
    print(f"=== SCHEMAS IN {CATALOG} ===")
    spark.sql(f"SHOW SCHEMAS IN {CATALOG}").show(80, truncate=False)
except Exception as e:
    print("SHOW SCHEMAS:", type(e).__name__, e)

In [ ]:
# USE SCHEMA shortcut (sets default for unqualified names in this session)
try:
    spark.sql(f"USE CATALOG {CATALOG}")
    spark.sql(f"USE SCHEMA {SCHEMA_NAME}")
    print(f"Default catalog.schema = {CATALOG}.{SCHEMA_NAME}")
except Exception as e:
    print("USE CATALOG/SCHEMA:", type(e).__name__, e)

## 3 — Register governed views over Day 3 Delta (ABFS)

### Why views, not tables?

| Approach | Pros | Cons |
|----------|------|------|
| **View on `delta.\`path\``** | No data copy; each student gets a governed `GRANT` target; same physical files | Slower than a managed table (reads raw path each time) |
| **External TABLE** | Registered in metastore with stats | Duplicate `LOCATION` blocked for shared paths; needs external location + admin |
| **Managed TABLE** | Full UC governance, stats, optimize | Copies data; more storage cost |

This lab uses **views** because the class **shares** one ADLS path.

### 3a — Silver view (full projection)

In [ ]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_silver_v")
spark.sql(
    f"CREATE VIEW {FULL_SCHEMA}.flights_silver_v "
    f"COMMENT 'Day 3 Silver — all columns, cleaned flight routes' "
    f"AS SELECT * FROM delta.`{DAY03_SILVER}`"
)
print("Created:", f"{FULL_SCHEMA}.flights_silver_v")
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_silver_v LIMIT 5").show(truncate=False)

### 3b — Gold view

In [ ]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_gold_v")
spark.sql(
    f"CREATE VIEW {FULL_SCHEMA}.flights_gold_v "
    f"COMMENT 'Day 3 Gold — total flights by destination' "
    f"AS SELECT * FROM delta.`{DAY03_GOLD}`"
)
print("Created:", f"{FULL_SCHEMA}.flights_gold_v")
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_gold_v ORDER BY total_flights DESC LIMIT 5").show(truncate=False)

### 3c — Sanity checks (row counts)

In [ ]:
silver_n = spark.sql(f"SELECT COUNT(*) AS n FROM {FULL_SCHEMA}.flights_silver_v").collect()[0]["n"]
gold_n   = spark.sql(f"SELECT COUNT(*) AS n FROM {FULL_SCHEMA}.flights_gold_v").collect()[0]["n"]
print(f"Silver rows: {silver_n}")
print(f"Gold rows:   {gold_n}")
assert silver_n > 0, "Silver is empty — check Day 3 paths"
assert gold_n > 0,   "Gold is empty — check Day 3 paths" 

### 3d — Why `SELECT *` in the view? Discussion

`SELECT *` passes through **all** columns from the Delta path. This is fine for:
- **Silver** (cleaned, not PII-heavy in this lab)
- **Gold** (aggregates — no individual records)

**In production**, prefer **explicit column lists** to:
1. Document the **contract** between view and consumers
2. Avoid exposing columns added later (schema evolution leakage)
3. Enable **column-level lineage** (UC tracks selected columns)

### 3e — **Projection view** (column minimization — governance pattern)

From **M-1/M-7 RBAC** labs: not every consumer needs **all** columns. A view can expose only safe fields.

**Scenario:** An analyst team only needs destination names and flight counts — not origin country or route keys.

In [ ]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_silver_dest_only_v")
spark.sql(
    "CREATE OR REPLACE VIEW "
    + f"{FULL_SCHEMA}.flights_silver_dest_only_v "
    + f"COMMENT 'Projection: destination + count only (column minimization)' "
    + f"AS SELECT DEST_COUNTRY_NAME, count FROM delta.`{DAY03_SILVER}`"
)
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_silver_dest_only_v LIMIT 8").show(truncate=False)
print("Columns:", spark.table(f"{FULL_SCHEMA}.flights_silver_dest_only_v").columns)

### 3f — **Filtered view** (row-level restriction pattern)

**Scenario:** A US-focused team should only see flights **to** the United States.

This is a **simple** version of Row-Level Security — a view with a `WHERE` clause.

In [ ]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_silver_us_only_v")
spark.sql(
    "CREATE OR REPLACE VIEW "
    + f"{FULL_SCHEMA}.flights_silver_us_only_v "
    + f"COMMENT 'Row filter: only US destinations (RLS lite)' "
    + f"AS SELECT * FROM delta.`{DAY03_SILVER}` WHERE DEST_COUNTRY_NAME = 'United States'"
)
us_count = spark.sql(f"SELECT COUNT(*) AS n FROM {FULL_SCHEMA}.flights_silver_us_only_v").collect()[0]["n"]
print(f"US-only view rows: {us_count}")
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_silver_us_only_v LIMIT 5").show(truncate=False)

### 3g — **Enriched / tagged view** (zone tagging)

**Scenario:** An enterprise catalog adds **constant columns** for zone, owner, or classification tags (useful in Catalog Explorer).

In [ ]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_silver_tagged_v")
spark.sql(
    "CREATE OR REPLACE VIEW "
    + f"{FULL_SCHEMA}.flights_silver_tagged_v "
    + f"COMMENT 'Enriched: zone tag + refresh timestamp' "
    + f"AS SELECT *, 'CURATED_SILVER' AS uc_zone_tag, current_timestamp() AS view_refreshed_at FROM delta.`{DAY03_SILVER}`"
)
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_silver_tagged_v LIMIT 3").show(truncate=False)

### 3h — **Masked view** (Tata Lab 16 pattern — simplified)

**Scenario:** Origin country is considered sensitive. Analysts see `***` instead of the real value.

(Real UC masking uses `ALTER TABLE ... SET MASKING POLICY` — shown in notebook **03** as theory.)

In [ ]:
spark.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.flights_silver_masked_v")
spark.sql(
    "CREATE OR REPLACE VIEW "
    + f"{FULL_SCHEMA}.flights_silver_masked_v "
    + f"COMMENT 'Masked: origin country hidden' "
    + f"AS SELECT DEST_COUNTRY_NAME, '***' AS ORIGIN_COUNTRY_NAME, count FROM delta.`{DAY03_SILVER}`"
)
spark.sql(f"SELECT * FROM {FULL_SCHEMA}.flights_silver_masked_v LIMIT 8").show(truncate=False)

### 3i — Summary of views created

| View | Pattern | Columns |
|------|---------|---------|
| `flights_silver_v` | Full projection | All Silver cols |
| `flights_gold_v` | Full projection | All Gold cols |
| `flights_silver_dest_only_v` | Column minimization | 2 cols only |
| `flights_silver_us_only_v` | Row filter (RLS lite) | All cols, US-dest only |
| `flights_silver_tagged_v` | Enriched (zone tag) | All + 2 tag cols |
| `flights_silver_masked_v` | Column masking | dest + masked origin + count |

Each demonstrates a **real governance pattern** you will encounter in production.

In [ ]:
# Count all views in your schema
spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").show(truncate=False)

## 4 — Physical vs logical: prove row parity

**Key insight:** The view is **metadata** in UC; the bytes live in **ADLS Delta**. Row counts should match.

In [ ]:
from pyspark.sql.functions import col

for label, path, view in [
    ("Silver", DAY03_SILVER, f"{FULL_SCHEMA}.flights_silver_v"),
    ("Gold",   DAY03_GOLD,   f"{FULL_SCHEMA}.flights_gold_v"),
]:
    n_delta = spark.read.format("delta").load(path).count()
    n_view  = spark.sql(f"SELECT COUNT(*) AS n FROM {view}").collect()[0]["n"]
    match   = "✓ MATCH" if n_delta == n_view else "✗ MISMATCH"
    print(f"{label}: delta={n_delta}  view={n_view}  {match}")

In [ ]:
# Schema comparison: Delta columns vs UC view columns
d_cols = set(spark.read.format("delta").load(DAY03_SILVER).columns)
v_cols = set(spark.table(f"{FULL_SCHEMA}.flights_silver_v").columns)
print("Delta cols:", sorted(d_cols))
print("View cols: ", sorted(v_cols))
print("Extra in Delta:", sorted(d_cols - v_cols))
print("Extra in view: ", sorted(v_cols - d_cols))

## 5 — Metadata exploration (`SHOW`, `DESCRIBE`, `DESCRIBE EXTENDED`)

SQL metadata commands are **your first debugging tool** when something goes wrong with a view or table.

In [ ]:
# 5a — SHOW TABLES (all objects in schema)
spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").show(truncate=False)

In [ ]:
# 5b — DESCRIBE (column names + types)
spark.sql(f"DESCRIBE {FULL_SCHEMA}.flights_silver_v").show(truncate=False)

In [ ]:
# 5c — DESCRIBE EXTENDED (ownership, view text, properties, etc.)
spark.sql(f"DESCRIBE EXTENDED {FULL_SCHEMA}.flights_silver_v").show(100, truncate=False)

In [ ]:
# 5d — DESCRIBE on projection view — notice fewer columns
spark.sql(f"DESCRIBE {FULL_SCHEMA}.flights_silver_dest_only_v").show(truncate=False)

In [ ]:
# 5e — DESCRIBE EXTENDED on Gold
spark.sql(f"DESCRIBE EXTENDED {FULL_SCHEMA}.flights_gold_v").show(100, truncate=False)

In [ ]:
# 5f — SHOW CREATE VIEW (view definition SQL — varies by DBR)
for v in ["flights_silver_v", "flights_silver_dest_only_v", "flights_silver_masked_v"]:
    try:
        print(f"=== {v} ===")
        spark.sql(f"SHOW CREATE TABLE {FULL_SCHEMA}.{v}").show(truncate=False)
    except Exception as e:
        print(f"  SHOW CREATE: {type(e).__name__}")

## 6 — Spark catalog API (programmatic discovery)

The `spark.catalog` Python API lets you **list** databases, tables, and columns without writing SQL — useful in orchestration and testing code.

In [ ]:
# 6a — List catalogs / databases visible to this session
try:
    dbs = spark.catalog.listDatabases()
    for d in dbs[:20]:
        print(d.name, "|", getattr(d, "description", ""))
except Exception as e:
    print("listDatabases:", type(e).__name__, e)

In [ ]:
# 6b — List tables in your schema
try:
    for t in spark.catalog.listTables(SCHEMA_NAME):
        print(f"  {t.name:40s} | type={getattr(t, 'tableType', '?')}")
except Exception as e:
    print("listTables:", type(e).__name__, e)
    print("Fallback → SHOW TABLES SQL")

In [ ]:
# 6c — List columns of a view
try:
    for col_meta in spark.catalog.listColumns(f"flights_silver_v", SCHEMA_NAME):
        print(f"  {col_meta.name:30s} | {col_meta.dataType}")
except Exception as e:
    print("listColumns:", type(e).__name__, e)

In [ ]:
# 6d — spark.table() = DataFrame from a UC name
silver_df = spark.table(f"{FULL_SCHEMA}.flights_silver_dest_only_v")
print("columns:", silver_df.columns)
print("count:", silver_df.count())
silver_df.show(5, truncate=False)

## 7 — Analytics SQL on governed views (10 queries)

These queries use **only** `main.day04_uXX_lab.*` names — identical to how an analyst would work post-`GRANT`.

### 7a — Top 10 destinations by total flights (Silver)

In [ ]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, SUM(count) AS flights FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 ORDER BY 2 DESC LIMIT 10""").show(truncate=False)

### 7b — Top 10 origin countries (Silver)

In [ ]:
spark.sql(f"""SELECT ORIGIN_COUNTRY_NAME, SUM(count) AS flights FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 ORDER BY 2 DESC LIMIT 10""").show(truncate=False)

### 7c — US–US route (domestic flights)

In [ ]:
spark.sql(f"""SELECT * FROM {FULL_SCHEMA}.flights_silver_v WHERE DEST_COUNTRY_NAME = 'United States' AND ORIGIN_COUNTRY_NAME = 'United States'""").show(truncate=False)

### 7d — Gold: max vs median total_flights

In [ ]:
spark.sql(f"""SELECT MAX(total_flights) AS max_flights, percentile_approx(total_flights, 0.5) AS median_flights, AVG(total_flights) AS avg_flights FROM {FULL_SCHEMA}.flights_gold_v""").show(truncate=False)

### 7e — High-volume destinations (HAVING)

In [ ]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, SUM(count) AS s FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 HAVING SUM(count) > 1000 ORDER BY 2 DESC LIMIT 15""").show(truncate=False)

### 7f — Distinct destination count

In [ ]:
spark.sql(f"""SELECT COUNT(DISTINCT DEST_COUNTRY_NAME) AS distinct_dest FROM {FULL_SCHEMA}.flights_silver_v""").show(truncate=False)

### 7g — Destinations starting with 'C'

In [ ]:
spark.sql(f"""SELECT DISTINCT DEST_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_silver_v WHERE DEST_COUNTRY_NAME LIKE 'C%' ORDER BY 1""").show(truncate=False)

### 7h — CASE bucket: high / medium / low volume routes

In [ ]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME, count, CASE WHEN count >= 1000 THEN 'high' WHEN count >= 100 THEN 'med' ELSE 'low' END AS bucket FROM {FULL_SCHEMA}.flights_silver_v LIMIT 20""").show(truncate=False)

### 7i — Routes with count between 100 and 500

In [ ]:
spark.sql(f"""SELECT * FROM {FULL_SCHEMA}.flights_silver_v WHERE count BETWEEN 100 AND 500 ORDER BY count DESC LIMIT 15""").show(truncate=False)

### 7j — US-only view: top 5 origin countries to US

In [ ]:
spark.sql(f"""SELECT ORIGIN_COUNTRY_NAME, SUM(count) AS to_us FROM {FULL_SCHEMA}.flights_silver_us_only_v GROUP BY 1 ORDER BY 2 DESC LIMIT 5""").show(truncate=False)

## 8 — DataFrame exercises on governed views

Same data, PySpark API. Privilege checks still apply (UC validates the name).

In [ ]:
from pyspark.sql.functions import col, sum as fsum, avg as favg, count as fcount, desc

# 8a — Gold: top 12 destinations
gdf = spark.table(f"{FULL_SCHEMA}.flights_gold_v")
gdf.orderBy(desc("total_flights")).select("DEST_COUNTRY_NAME", "total_flights").show(12, truncate=False)

In [ ]:
# 8b — Silver: outbound flights per origin
spark.table(f"{FULL_SCHEMA}.flights_silver_v") \
    .groupBy("ORIGIN_COUNTRY_NAME") \
    .agg(fsum("count").alias("outbound")) \
    .orderBy(desc("outbound")) \
    .show(10, truncate=False)

In [ ]:
# 8c — Projection view: average count per destination
spark.table(f"{FULL_SCHEMA}.flights_silver_dest_only_v") \
    .groupBy("DEST_COUNTRY_NAME") \
    .agg(favg("count").alias("avg_count"), fcount("count").alias("routes")) \
    .orderBy(desc("avg_count")) \
    .show(12, truncate=False)

In [ ]:
# 8d — Tagged view: check enrichment columns
spark.table(f"{FULL_SCHEMA}.flights_silver_tagged_v") \
    .select("DEST_COUNTRY_NAME", "uc_zone_tag", "view_refreshed_at") \
    .limit(5).show(truncate=False)

In [ ]:
# 8e — Masked view: prove origin is hidden
spark.table(f"{FULL_SCHEMA}.flights_silver_masked_v") \
    .select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count") \
    .show(8, truncate=False)

## 9 — Joins, subqueries, window functions (governed views)

### 9a — Join Silver + Gold on destination

In [ ]:
spark.sql(f"""
SELECT s.DEST_COUNTRY_NAME, SUM(s.count) AS silver_sum, MAX(g.total_flights) AS gold_total
FROM {FULL_SCHEMA}.flights_silver_v s
JOIN {FULL_SCHEMA}.flights_gold_v g ON s.DEST_COUNTRY_NAME = g.DEST_COUNTRY_NAME
GROUP BY s.DEST_COUNTRY_NAME
ORDER BY silver_sum DESC LIMIT 12
""").show(truncate=False)

### 9b — Subquery EXISTS (routes to high-volume destinations)

In [ ]:
spark.sql(f"""
SELECT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME, count
FROM {FULL_SCHEMA}.flights_silver_v s
WHERE EXISTS (
  SELECT 1 FROM {FULL_SCHEMA}.flights_gold_v g
  WHERE g.DEST_COUNTRY_NAME = s.DEST_COUNTRY_NAME AND g.total_flights > 5000
)
ORDER BY count DESC LIMIT 20
""").show(truncate=False)

### 9c — Window: rank destinations by total_flights

In [ ]:
spark.sql(f"""
SELECT DEST_COUNTRY_NAME, total_flights,
       ROW_NUMBER() OVER (ORDER BY total_flights DESC) AS rnk,
       NTILE(4) OVER (ORDER BY total_flights DESC) AS quartile
FROM {FULL_SCHEMA}.flights_gold_v
""").where("rnk <= 15").show(truncate=False)

### 9d — Window: top 3 routes per destination (Silver)

In [ ]:
spark.sql(f"""
SELECT * FROM (
  SELECT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME, count,
         ROW_NUMBER() OVER (PARTITION BY DEST_COUNTRY_NAME ORDER BY count DESC) AS rnk
  FROM {FULL_SCHEMA}.flights_silver_v
) t WHERE rnk <= 3
ORDER BY DEST_COUNTRY_NAME, rnk
LIMIT 40
""").show(truncate=False)

### 9e — CTE: destinations above average Gold total_flights

In [ ]:
spark.sql(f"""
WITH avg_tbl AS (SELECT AVG(total_flights) AS avg_f FROM {FULL_SCHEMA}.flights_gold_v)
SELECT g.DEST_COUNTRY_NAME, g.total_flights, ROUND(a.avg_f, 1) AS avg_flights
FROM {FULL_SCHEMA}.flights_gold_v g
CROSS JOIN avg_tbl a
WHERE g.total_flights > a.avg_f
ORDER BY g.total_flights DESC
""").show(truncate=False)

### 9f — INTERSECT: destinations present in both Silver and Gold

In [ ]:
spark.sql(f"""
SELECT DISTINCT DEST_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_silver_v
INTERSECT
SELECT DISTINCT DEST_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_gold_v
ORDER BY 1 LIMIT 25
""").show(truncate=False)

### 9g — LEFT ANTI JOIN: Gold destinations NOT in Silver (should be empty or rare)

In [ ]:
spark.sql(f"""
SELECT g.DEST_COUNTRY_NAME
FROM {FULL_SCHEMA}.flights_gold_v g
LEFT ANTI JOIN {FULL_SCHEMA}.flights_silver_v s
  ON g.DEST_COUNTRY_NAME = s.DEST_COUNTRY_NAME
LIMIT 20
""").show(truncate=False)

## 10 — `EXPLAIN` (query plan for a governed view)

Check whether **predicate pushdown** reaches the Delta scan layer (look for `PushedFilters`).

In [ ]:
spark.sql(f"EXPLAIN EXTENDED SELECT * FROM {FULL_SCHEMA}.flights_silver_v WHERE DEST_COUNTRY_NAME = 'Germany'").show(truncate=False)

In [ ]:
# Compare: projection view should scan fewer columns
spark.sql(f"EXPLAIN SELECT * FROM {FULL_SCHEMA}.flights_silver_dest_only_v WHERE DEST_COUNTRY_NAME = 'Germany'").show(truncate=False)

## 11 — `hive_metastore` peek (optional)

Listing legacy schemas builds a **mental map** of what lives where. Safe to skip if error.

In [ ]:
try:
    spark.sql("SHOW SCHEMAS IN hive_metastore").show(30, truncate=False)
except Exception as e:
    print("hive_metastore:", type(e).__name__, str(e)[:120])

## 12 — Temp view vs permanent UC view

| Kind | Lifetime | `GRANT`-able? |
|------|----------|---------------|
| **`CREATE TEMPORARY VIEW`** | Spark session only | No |
| **`CREATE VIEW` in UC** | Metastore-persisted | Yes |

In [ ]:
# Temp view: useful for ad-hoc exploration, not for sharing
spark.table(f"{FULL_SCHEMA}.flights_silver_v").where("count > 500").createOrReplaceTempView("tmp_high_vol")
spark.sql("SELECT COUNT(*) AS n FROM tmp_high_vol").show()
spark.sql("SELECT * FROM tmp_high_vol ORDER BY count DESC LIMIT 6").show(truncate=False)

In [ ]:
# Confirm temp view is NOT in UC
spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").show(truncate=False)
# tmp_high_vol does NOT appear — it lives in the session only

## 13 — `COMMENT ON VIEW` (documentation in Catalog Explorer)

In [ ]:
for v, cmnt in [
    ("flights_gold_v", "Gold KPI by destination — Day 3 pipeline; Day 4 UC view."),
    ("flights_silver_us_only_v", "Row-filtered view: only US destinations (RLS lite)."),
]:
    try:
        spark.sql(f"COMMENT ON TABLE {FULL_SCHEMA}.{v} IS '{cmnt}'")
        print(f"Comment set on {v}")
    except Exception as e:
        print(f"COMMENT ON {v}:", type(e).__name__)

## 14 — Delta storage metadata (`DESCRIBE DETAIL`, `DESCRIBE HISTORY`)

The **physical** Delta table behind the view has its own metadata. These commands work on the **path**, not the view name.

In [ ]:
# DESCRIBE DETAIL — file count, size, partitioning
for label, path in [("Silver", DAY03_SILVER), ("Gold", DAY03_GOLD)]:
    print(f"=== {label} DETAIL ===")
    spark.sql(f"DESCRIBE DETAIL delta.`{path}`").select("format", "numFiles", "sizeInBytes", "partitionColumns").show(truncate=False)

In [ ]:
# DESCRIBE HISTORY — transaction log
spark.sql(f"DESCRIBE HISTORY delta.`{DAY03_SILVER}`").select("version", "timestamp", "operation").show(10, truncate=False)

In [ ]:
# Extract numFiles programmatically
nf = spark.sql(f"DESCRIBE DETAIL delta.`{DAY03_GOLD}`").select("numFiles").collect()[0]["numFiles"]
print(f"Gold Delta numFiles = {nf}")

## 15 — Mini challenges

Complete these in the empty code cells. Compare approaches with others in the class or review in a follow-up session.

### C1 — Write a CTE that ranks Gold destinations and returns only **top 5** with `ROW_NUMBER()`.

In [ ]:
# Your solution:


### C2 — Count **distinct** `ORIGIN_COUNTRY_NAME` in Silver using `COUNT(DISTINCT ...)`.

In [ ]:
# Your solution:


### C3 — Join `flights_silver_us_only_v` with `flights_gold_v` on `DEST_COUNTRY_NAME` — does `United States` appear?

In [ ]:
# Your solution:


### C4 — Using `DESCRIBE DETAIL` on `DAY03_GOLD`, extract `sizeInBytes` into a Python variable and print it in MB.

In [ ]:
# Your solution:


### C5 — Explain in one sentence: **Why can two students both `SELECT` the same ADLS path but still need different UC schemas?**

In [ ]:
# Your solution:


### C6 — Create a view `flights_gold_top10_v` that only exposes the **top 10** destinations (use a subquery or CTE).

In [ ]:
# Your solution:


## 16 — SQL skill builders (10 drills — run each cell, tweak filters)

### 16a — Average count per destination (top 12)

In [ ]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, ROUND(AVG(count),1) AS avg_cnt FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 ORDER BY 2 DESC LIMIT 12""").show(truncate=False)

### 16b — Median count per origin country

In [ ]:
spark.sql(f"""SELECT ORIGIN_COUNTRY_NAME, percentile_approx(count, 0.5) AS med FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 ORDER BY 2 DESC LIMIT 12""").show(truncate=False)

### 16c — Pair count: how many distinct (dest, origin) pairs?

In [ ]:
spark.sql(f"""SELECT COUNT(*) AS pairs FROM (SELECT DISTINCT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_silver_v)""").show(truncate=False)

### 16d — Gold: destinations above 90th percentile

In [ ]:
spark.sql(f"""SELECT * FROM {FULL_SCHEMA}.flights_gold_v WHERE total_flights >= (SELECT percentile_approx(total_flights, 0.9) FROM {FULL_SCHEMA}.flights_gold_v) ORDER BY total_flights DESC""").show(truncate=False)

### 16e — Compare sum(Silver) vs sum(Gold)

In [ ]:
spark.sql(f"""SELECT (SELECT SUM(count) FROM {FULL_SCHEMA}.flights_silver_v) AS silver_sum, (SELECT SUM(total_flights) FROM {FULL_SCHEMA}.flights_gold_v) AS gold_sum""").show(truncate=False)

### 16f — Silver: countries with exactly 1 route

In [ ]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, COUNT(*) AS routes FROM {FULL_SCHEMA}.flights_silver_v GROUP BY 1 HAVING COUNT(*) = 1 ORDER BY 1""").show(truncate=False)

### 16g — Gold: ratio of each destination to grand total

In [ ]:
spark.sql(f"""SELECT DEST_COUNTRY_NAME, total_flights, ROUND(total_flights * 100.0 / (SELECT SUM(total_flights) FROM {FULL_SCHEMA}.flights_gold_v), 2) AS pct FROM {FULL_SCHEMA}.flights_gold_v ORDER BY pct DESC LIMIT 15""").show(truncate=False)

### 16h — Masked view: aggregate on masked data (prove it works)

In [ ]:
spark.sql(f"""SELECT ORIGIN_COUNTRY_NAME, SUM(count) AS s FROM {FULL_SCHEMA}.flights_silver_masked_v GROUP BY 1""").show(truncate=False)

### 16i — Silver: MIN, MAX, STDDEV of count

In [ ]:
spark.sql(f"""SELECT MIN(count) AS min_c, MAX(count) AS max_c, ROUND(STDDEV(count),1) AS std_c FROM {FULL_SCHEMA}.flights_silver_v""").show(truncate=False)

### 16j — UNION destinations from Silver + Gold (deduplicated)

In [ ]:
spark.sql(f"""SELECT DISTINCT DEST_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_silver_v UNION SELECT DISTINCT DEST_COUNTRY_NAME FROM {FULL_SCHEMA}.flights_gold_v ORDER BY 1""").show(truncate=False)

## Next

Continue with `03-Day4-Unity-Catalog-Security-Lineage.ipynb` (grants, RBAC, metadata views, lineage, audit).

Cleanup (optional): drop your lab schema only if class policy allows.

```python
# spark.sql(f"DROP SCHEMA IF EXISTS {FULL_SCHEMA} CASCADE")
```